# House Prices: Stacked Ensemble Regression Pipeline

**Competition:** House Prices - Advanced Regression Techniques

**Notebook Focus:** Multi-model stacking, domain-driven feature engineering, RMSLE-optimized ensemble

**Author:** [Amey Thakur](https://www.kaggle.com/ameythakur20)

---

## Workspace Navigation
1. Data Acquisition
2. Data Inspection
3. Data Cleaning
4. EDA
5. Feature Engineering
6. Modeling
7. Evaluation
8. Conclusion
9. References

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from scipy.stats import skew, norm
from scipy.special import boxcox1p

from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.kernel_ridge import KernelRidge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_squared_error

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# Premium aesthetic configuration
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'axes.titleweight': 'bold',
    'axes.spines.top': False,
    'axes.spines.right': False
})

pd.DataFrame([{'Library': lib, 'Status': '[LOADED]'} for lib in
    ['pandas', 'numpy', 'matplotlib', 'seaborn', 'scipy', 'sklearn',
     'xgboost', 'lightgbm']])

## 1. Data Acquisition
Loading the competition datasets from the standard Kaggle input directory.

In [ ]:
ROOT = '/kaggle/input/competitions/house-prices-advanced-regression-techniques'

train = pd.read_csv(f'{ROOT}/train.csv')
test  = pd.read_csv(f'{ROOT}/test.csv')

# Preserve identifiers and target
train_id = train['Id']
test_id  = test['Id']
y_train  = train['SalePrice']

# Combine for unified preprocessing
ntrain = train.shape[0]
ntest  = test.shape[0]

all_data = pd.concat([train, test], sort=False).reset_index(drop=True)
all_data.drop(['Id', 'SalePrice'], axis=1, inplace=True)

pd.DataFrame([
    {'Dataset': 'Train',    'Rows': ntrain,           'Cols': train.shape[1]},
    {'Dataset': 'Test',     'Rows': ntest,            'Cols': test.shape[1]},
    {'Dataset': 'Combined', 'Rows': all_data.shape[0],'Cols': all_data.shape[1]}
])

## 2. Data Inspection
Profiling missing value densities, data type distributions, and target variable characteristics.

In [ ]:
# Missing value analysis
missing = all_data.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing_pct = (missing / len(all_data) * 100).round(2)

fig, ax = plt.subplots(figsize=(14, 6))
sns.barplot(x=missing_pct.index[:20], y=missing_pct.values[:20], color='#e377c2', alpha=0.9, ax=ax)
ax.set_title('Top 20 Features by Missing Value Percentage')
ax.set_ylabel('Missing (%)')
ax.tick_params(axis='x', rotation=60)
plt.tight_layout()
plt.show()

pd.DataFrame({'Feature': missing.index[:20], 'Missing': missing.values[:20],
              'Percentage': missing_pct.values[:20]})

In [ ]:
# Data type distribution
dtype_counts = all_data.dtypes.value_counts()
fig, ax = plt.subplots(figsize=(6, 4))
dtype_counts.plot(kind='bar', color=['#1f77b4', '#ff7f0e', '#2ca02c'], ax=ax)
ax.set_title('Feature Data Type Distribution')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

pd.DataFrame([
    {'Type': str(k), 'Count': v} for k, v in dtype_counts.items()
])

## 3. Data Cleaning
Handling missing values with domain-specific imputation strategies. Outlier removal targets known anomalous observations in the training set.

In [ ]:
# Remove outliers identified in competition literature
# GrLivArea > 4000 with low SalePrice are known anomalous data points
outlier_idx = train[(train['GrLivArea'] > 4000) & (train['SalePrice'] < 300000)].index

fig, ax = plt.subplots(figsize=(10, 6))
sns.scatterplot(data=train, x='GrLivArea', y='SalePrice', color='#2ca02c', alpha=0.5, s=40, ax=ax)
ax.scatter(train.loc[outlier_idx, 'GrLivArea'], train.loc[outlier_idx, 'SalePrice'],
           c='red', s=100, marker='x', linewidths=2, label='Outliers (removed)')
ax.set_title('GrLivArea vs SalePrice: Outlier Identification')
ax.legend()
plt.show()

# Apply removal
train = train.drop(outlier_idx)
y_train = train['SalePrice']
ntrain = train.shape[0]

# Rebuild combined dataset after outlier removal
all_data = pd.concat([train.drop(['Id', 'SalePrice'], axis=1),
                      test.drop(['Id'], axis=1)], sort=False).reset_index(drop=True)

pd.DataFrame([{'Action': 'Outliers Removed', 'Count': len(outlier_idx), 'Status': '[SUCCESS]'}])

In [ ]:
# Log-transform target to reduce skewness
y_train = np.log1p(y_train.values)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(np.expm1(y_train), kde=True, color='#1f77b4', bins=40, ax=axes[0])
axes[0].set_title('SalePrice (Original)')
sns.histplot(y_train, kde=True, color='#ff7f0e', bins=40, ax=axes[1])
axes[1].set_title('SalePrice (Log1p Transformed)')
plt.tight_layout()
plt.show()

In [ ]:
# Domain-specific missing value imputation

# Features where NaN means "None" (no feature present)
none_cols = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu',
             'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
             'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
             'MasVnrType', 'MSSubClass']
for col in none_cols:
    if col in all_data.columns:
        all_data[col] = all_data[col].fillna('None')

# Features where NaN means 0 (no area/count)
zero_cols = ['GarageYrBlt', 'GarageArea', 'GarageCars',
             'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF',
             'BsmtFullBath', 'BsmtHalfBath', 'MasVnrArea']
for col in zero_cols:
    if col in all_data.columns:
        all_data[col] = all_data[col].fillna(0)

# Mode imputation for categorical features
mode_cols = ['MSZoning', 'Electrical', 'KitchenQual', 'Exterior1st',
             'Exterior2nd', 'SaleType', 'Functional']
for col in mode_cols:
    if col in all_data.columns:
        all_data[col] = all_data[col].fillna(all_data[col].mode()[0])

# LotFrontage: impute by neighborhood median
if 'LotFrontage' in all_data.columns:
    all_data['LotFrontage'] = all_data.groupby('Neighborhood')['LotFrontage'].transform(
        lambda x: x.fillna(x.median()))

# Utilities: near-zero variance, drop
if 'Utilities' in all_data.columns:
    all_data.drop(['Utilities'], axis=1, inplace=True)

# Fill any remaining NaN
all_data = all_data.fillna(0)

remaining_nulls = all_data.isnull().sum().sum()
pd.DataFrame([
    {'Action': 'Domain Imputation Complete', 'Remaining Nulls': remaining_nulls, 'Status': '[SUCCESS]'}
])

## 4. EDA
Visualizing feature correlations, categorical distributions, and spatial relationships across the cleaned dataset.

### 4.1 Pearson Correlation Heatmap

In [ ]:
# Correlation matrix on training subset
train_subset = all_data.iloc[:ntrain].copy()
train_subset['SalePrice'] = np.expm1(y_train)
numeric_feats = train_subset.select_dtypes(include=[np.number])
corr = numeric_feats.corr()
top_corr = corr['SalePrice'].sort_values(key=abs, ascending=False).head(11).index

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(train_subset[top_corr].corr(), annot=True, cmap='coolwarm', fmt='.2f',
            square=True, linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Top 10 Feature Pearson Correlation Matrix')
plt.tight_layout()
plt.show()

### 4.2 Continuous Feature Scatter Projections

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

sns.scatterplot(data=train_subset, x='GrLivArea', y='SalePrice', hue='OverallQual',
                palette='viridis', alpha=0.7, s=50, ax=axes[0, 0])
axes[0, 0].set_title('Living Area vs SalePrice (by Quality)')

sns.scatterplot(data=train_subset, x='TotalBsmtSF', y='SalePrice',
                color='#2ca02c', alpha=0.5, s=40, ax=axes[0, 1])
axes[0, 1].set_title('Total Basement SF vs SalePrice')

sns.scatterplot(data=train_subset, x='YearBuilt', y='SalePrice',
                color='#9467bd', alpha=0.5, s=40, ax=axes[1, 0])
axes[1, 0].set_title('Year Built vs SalePrice')

sns.boxplot(data=train_subset, x='GarageCars', y='SalePrice',
            color='#8c564b', ax=axes[1, 1])
axes[1, 1].set_title('Garage Capacity vs SalePrice')

plt.tight_layout()
plt.show()

### 4.3 Neighborhood Variance

In [ ]:
fig, ax = plt.subplots(figsize=(16, 7))
sorted_nb = train_subset.groupby('Neighborhood')['SalePrice'].median().sort_values().index
sns.boxplot(data=train_subset, x='Neighborhood', y='SalePrice',
            order=sorted_nb, color='#8c564b', fliersize=3, ax=ax)
ax.set_title('SalePrice Variance by Neighborhood')
ax.tick_params(axis='x', rotation=90)
plt.tight_layout()
plt.show()

### 4.4 Quality and Temporal Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

sns.violinplot(data=train_subset, x='OverallQual', y='SalePrice', color='#bcbd22', ax=axes[0])
axes[0].set_title('SalePrice by Overall Quality')

year_med = train_subset.groupby('YearBuilt')['SalePrice'].median()
axes[1].plot(year_med.index, year_med.values, color='#1f77b4', linewidth=1.5)
axes[1].fill_between(year_med.index, year_med.values, alpha=0.15, color='#1f77b4')
axes[1].set_title('Median SalePrice by Construction Year')
axes[1].set_xlabel('Year Built')

plt.tight_layout()
plt.show()

del train_subset

## 5. Feature Engineering
Constructing domain-driven composite features, encoding ordinal quality scales, and applying skewness correction to continuous variables.

In [ ]:
# Composite area features
all_data['TotalSF']       = all_data['TotalBsmtSF'] + all_data['1stFlrSF'] + all_data['2ndFlrSF']
all_data['TotalPorchSF']  = (all_data['OpenPorchSF'] + all_data['EnclosedPorch'] +
                             all_data['3SsnPorch'] + all_data['ScreenPorch'] + all_data['WoodDeckSF'])
all_data['TotalBath']     = (all_data['FullBath'] + 0.5 * all_data['HalfBath'] +
                             all_data['BsmtFullBath'] + 0.5 * all_data['BsmtHalfBath'])
all_data['HasPool']       = (all_data['PoolArea'] > 0).astype(int)
all_data['Has2ndFloor']   = (all_data['2ndFlrSF'] > 0).astype(int)
all_data['HasGarage']     = (all_data['GarageArea'] > 0).astype(int)
all_data['HasBsmt']       = (all_data['TotalBsmtSF'] > 0).astype(int)
all_data['HasFireplace']  = (all_data['Fireplaces'] > 0).astype(int)
all_data['WasRemodeled']  = (all_data['YearRemodAdd'] != all_data['YearBuilt']).astype(int)
all_data['HouseAge']      = all_data['YrSold'] - all_data['YearBuilt']
all_data['RemodAge']      = all_data['YrSold'] - all_data['YearRemodAdd']

pd.DataFrame([
    {'Feature': 'TotalSF',      'Formula': 'Bsmt + 1st + 2nd',                 'Status': '[SUCCESS]'},
    {'Feature': 'TotalPorchSF', 'Formula': 'Open + Enclosed + 3Ssn + Screen + Wood', 'Status': '[SUCCESS]'},
    {'Feature': 'TotalBath',    'Formula': 'Full + 0.5*Half + BsmtFull + 0.5*BsmtHalf', 'Status': '[SUCCESS]'},
    {'Feature': 'HouseAge',     'Formula': 'YrSold - YearBuilt',               'Status': '[SUCCESS]'},
    {'Feature': 'RemodAge',     'Formula': 'YrSold - YearRemodAdd',             'Status': '[SUCCESS]'},
    {'Feature': 'Binary Flags', 'Formula': 'Pool, 2ndFloor, Garage, Bsmt, FP, Remod', 'Status': '[SUCCESS]'}
])

In [ ]:
# Ordinal quality encoding
quality_map = {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}
ordinal_cols = ['ExterQual', 'ExterCond', 'HeatingQC', 'KitchenQual',
                'BsmtQual', 'BsmtCond', 'FireplaceQu', 'GarageQual',
                'GarageCond', 'PoolQC']
for col in ordinal_cols:
    if col in all_data.columns:
        all_data[col] = all_data[col].map(quality_map).fillna(0).astype(int)

# BsmtExposure encoding
bsmt_exp_map = {'None': 0, 'No': 1, 'Mn': 2, 'Av': 3, 'Gd': 4}
if 'BsmtExposure' in all_data.columns:
    all_data['BsmtExposure'] = all_data['BsmtExposure'].map(bsmt_exp_map).fillna(0).astype(int)

# BsmtFinType encoding
bsmt_fin_map = {'None': 0, 'Unf': 1, 'LwQ': 2, 'Rec': 3, 'BLQ': 4, 'ALQ': 5, 'GLQ': 6}
for col in ['BsmtFinType1', 'BsmtFinType2']:
    if col in all_data.columns:
        all_data[col] = all_data[col].map(bsmt_fin_map).fillna(0).astype(int)

# GarageFinish encoding
gar_fin_map = {'None': 0, 'Unf': 1, 'RFn': 2, 'Fin': 3}
if 'GarageFinish' in all_data.columns:
    all_data['GarageFinish'] = all_data['GarageFinish'].map(gar_fin_map).fillna(0).astype(int)

# Fence encoding
fence_map = {'None': 0, 'MnWw': 1, 'GdWo': 2, 'MnPrv': 3, 'GdPrv': 4}
if 'Fence' in all_data.columns:
    all_data['Fence'] = all_data['Fence'].map(fence_map).fillna(0).astype(int)

pd.DataFrame([{'Action': 'Ordinal Encoding Complete', 'Columns Encoded': len(ordinal_cols) + 4, 'Status': '[SUCCESS]'}])

In [ ]:
# Skewness correction on numeric features using Box-Cox transform
numeric_feats = all_data.dtypes[all_data.dtypes != 'object'].index
skewed_feats = all_data[numeric_feats].apply(lambda x: skew(x.dropna())).sort_values(ascending=False)
high_skew = skewed_feats[abs(skewed_feats) > 0.75]

fig, ax = plt.subplots(figsize=(14, 5))
high_skew.head(20).plot(kind='bar', color='#17becf', ax=ax)
ax.set_title('Top 20 Skewed Features (Before Correction)')
ax.set_ylabel('Skewness')
plt.tight_layout()
plt.show()

# Apply Box-Cox transformation
lam = 0.15
for feat in high_skew.index:
    all_data[feat] = boxcox1p(all_data[feat], lam)

pd.DataFrame([{'Action': 'Box-Cox Correction', 'Features Corrected': len(high_skew), 'Lambda': lam, 'Status': '[SUCCESS]'}])

In [ ]:
# One-hot encode remaining categorical features
all_data = pd.get_dummies(all_data)

pd.DataFrame([
    {'Action': 'One-Hot Encoding',   'Final Feature Count': all_data.shape[1], 'Status': '[SUCCESS]'},
    {'Action': 'Train/Test Split',   'Train Rows': ntrain, 'Test Rows': ntest, 'Status': '[SUCCESS]'}
])

## 6. Modeling
Training a stacked ensemble of six base learners with a Ridge meta-learner. Each model is validated using stratified 5-fold cross-validation with RMSLE scoring.

In [ ]:
# Split back to train and test
X_train = all_data[:ntrain]
X_test  = all_data[ntrain:]

# Cross-validation setup
n_folds = 5
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)

def rmsle_cv(model, X=X_train, y=y_train):
    rmse = np.sqrt(-cross_val_score(model, X, y, scoring='neg_mean_squared_error', cv=kf))
    return rmse

pd.DataFrame([{'Setup': 'Cross-Validation', 'Folds': n_folds, 'Metric': 'RMSLE', 'Status': '[READY]'}])

In [ ]:
# ============================================================================
# BASE LEARNER 1: Ridge Regression
# ============================================================================
ridge = make_pipeline(RobustScaler(), Ridge(alpha=15.0, random_state=42))
score = rmsle_cv(ridge)
print(f"Ridge          : {score.mean():.5f} (+/- {score.std():.5f})")

# ============================================================================
# BASE LEARNER 2: Lasso Regression
# ============================================================================
lasso = make_pipeline(RobustScaler(), Lasso(alpha=0.0005, random_state=42, max_iter=10000))
score = rmsle_cv(lasso)
print(f"Lasso          : {score.mean():.5f} (+/- {score.std():.5f})")

# ============================================================================
# BASE LEARNER 3: ElasticNet
# ============================================================================
enet = make_pipeline(RobustScaler(), ElasticNet(alpha=0.0005, l1_ratio=0.9, random_state=42, max_iter=10000))
score = rmsle_cv(enet)
print(f"ElasticNet     : {score.mean():.5f} (+/- {score.std():.5f})")

# ============================================================================
# BASE LEARNER 4: Kernel Ridge
# ============================================================================
krr = KernelRidge(alpha=0.6, kernel='polynomial', degree=2, coef0=2.5)
score = rmsle_cv(krr)
print(f"Kernel Ridge   : {score.mean():.5f} (+/- {score.std():.5f})")

# ============================================================================
# BASE LEARNER 5: Gradient Boosting
# ============================================================================
gbr = GradientBoostingRegressor(
    n_estimators=3000, learning_rate=0.01, max_depth=4,
    max_features='sqrt', min_samples_leaf=15, min_samples_split=10,
    loss='huber', random_state=42
)
score = rmsle_cv(gbr)
print(f"GradientBoost  : {score.mean():.5f} (+/- {score.std():.5f})")

# ============================================================================
# BASE LEARNER 6: XGBoost
# ============================================================================
xgb = XGBRegressor(
    colsample_bytree=0.4, gamma=0.05, learning_rate=0.01,
    max_depth=3, min_child_weight=1.8, n_estimators=3000,
    reg_alpha=0.5, reg_lambda=0.8, subsample=0.5,
    random_state=42, verbosity=0
)
score = rmsle_cv(xgb)
print(f"XGBoost        : {score.mean():.5f} (+/- {score.std():.5f})")

# ============================================================================
# BASE LEARNER 7: LightGBM
# ============================================================================
lgbm = LGBMRegressor(
    objective='regression', num_leaves=5, learning_rate=0.01,
    n_estimators=3000, max_bin=55, bagging_fraction=0.8,
    bagging_freq=5, feature_fraction=0.23, feature_fraction_seed=9,
    bagging_seed=9, min_data_in_leaf=6, min_sum_hessian_in_leaf=11,
    verbose=-1, random_state=42
)
score = rmsle_cv(lgbm)
print(f"LightGBM       : {score.mean():.5f} (+/- {score.std():.5f})")

In [ ]:
# ============================================================================
# STACKED ENSEMBLE: Out-of-fold predictions as meta-features
# ============================================================================

base_models = {
    'Ridge': ridge,
    'Lasso': lasso,
    'ElasticNet': enet,
    'KernelRidge': krr,
    'GBR': gbr,
    'XGBoost': xgb,
    'LightGBM': lgbm
}

# Generate out-of-fold predictions
oof_train = np.zeros((ntrain, len(base_models)))
oof_test  = np.zeros((ntest, len(base_models)))

for idx, (name, model) in enumerate(base_models.items()):
    oof_test_folds = np.zeros((ntest, n_folds))

    for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X_train)):
        X_tr = X_train.iloc[train_idx]
        y_tr = y_train[train_idx]
        X_val = X_train.iloc[val_idx]

        model.fit(X_tr, y_tr)
        oof_train[val_idx, idx] = model.predict(X_val)
        oof_test_folds[:, fold_idx] = model.predict(X_test)

    oof_test[:, idx] = oof_test_folds.mean(axis=1)
    oof_score = np.sqrt(mean_squared_error(y_train, oof_train[:, idx]))
    print(f"[STACKED] {name:14s} OOF RMSLE: {oof_score:.5f}")

pd.DataFrame([{'Stage': 'OOF Generation', 'Models': len(base_models), 'Status': '[SUCCESS]'}])

In [ ]:
# Meta-learner: Ridge on stacked features
meta_model = Ridge(alpha=6.0)
meta_model.fit(oof_train, y_train)
stacked_pred = meta_model.predict(oof_test)

# Visualize meta-learner weights
fig, ax = plt.subplots(figsize=(10, 5))
weights = meta_model.coef_
model_names = list(base_models.keys())
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#17becf']
bars = ax.bar(model_names, weights, color=colors, alpha=0.85)
ax.set_title('Meta-Learner (Ridge) Weight Distribution Across Base Models')
ax.set_ylabel('Coefficient Weight')
ax.tick_params(axis='x', rotation=30)
for bar, w in zip(bars, weights):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
            f'{w:.3f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Direct blend as an alternative (weighted average of top models)
# Fit all base models on full training data for direct predictions
for name, model in base_models.items():
    model.fit(X_train, y_train)

direct_preds = {}
for name, model in base_models.items():
    direct_preds[name] = model.predict(X_test)

# Weighted blend (empirically tuned weights)
blended_pred = (
    0.10 * direct_preds['Ridge'] +
    0.10 * direct_preds['Lasso'] +
    0.10 * direct_preds['ElasticNet'] +
    0.05 * direct_preds['KernelRidge'] +
    0.20 * direct_preds['GBR'] +
    0.20 * direct_preds['XGBoost'] +
    0.25 * direct_preds['LightGBM']
)

# Final ensemble: average of stacked and blended
final_pred = 0.5 * stacked_pred + 0.5 * blended_pred

pd.DataFrame([
    {'Ensemble': 'Stacked (Ridge meta)', 'Weight': 0.5},
    {'Ensemble': 'Blended (weighted avg)', 'Weight': 0.5},
    {'Ensemble': 'Final (combined)', 'Weight': 1.0, 'Status': '[SUCCESS]'}
])

## 7. Evaluation
Analyzing the final ensemble predictions and staging the submission artifact.

In [ ]:
# Inverse log transform to get dollar predictions
final_prices = np.expm1(final_pred)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Predicted distribution
sns.histplot(final_prices, kde=True, color='#17becf', bins=40, ax=axes[0])
axes[0].set_title('Predicted SalePrice Distribution')
axes[0].set_xlabel('Predicted SalePrice ($)')

# Train vs predicted overlay
sns.kdeplot(np.expm1(y_train), color='#1f77b4', label='Train (Actual)', ax=axes[1])
sns.kdeplot(final_prices, color='#17becf', label='Test (Predicted)', ax=axes[1])
axes[1].set_title('Train Actual vs Test Predicted Overlay')
axes[1].legend()

plt.tight_layout()
plt.show()

pd.DataFrame([
    {'Stat': 'Min Prediction',    'Value': f"${final_prices.min():,.0f}"},
    {'Stat': 'Max Prediction',    'Value': f"${final_prices.max():,.0f}"},
    {'Stat': 'Mean Prediction',   'Value': f"${final_prices.mean():,.0f}"},
    {'Stat': 'Median Prediction', 'Value': f"${np.median(final_prices):,.0f}"}
])

In [ ]:
# Generate submission
submission = pd.DataFrame({'Id': test_id, 'SalePrice': final_prices})
submission.to_csv('submission.csv', index=False)

display(submission.head(10))

pd.DataFrame([
    {'Operation': 'submission.csv generated', 'Records': len(submission), 'Status': '[SUCCESS]'}
])

## 8. Conclusion
- A 7-model stacked ensemble (Ridge, Lasso, ElasticNet, Kernel Ridge, Gradient Boosting, XGBoost, LightGBM) provides robust generalization by aggregating diverse learning paradigms.
- Log1p transformation of the target and Box-Cox correction of skewed predictors significantly reduce heteroscedasticity and improve linear model performance.
- Domain-specific missing value imputation (NaN as "None" for categorical features, neighborhood-median for LotFrontage) preserves information that generic imputation would destroy.
- Composite features (TotalSF, TotalBath, HouseAge, TotalPorchSF) and ordinal quality encoding capture nonlinear structural relationships in the Ames housing market.
- Outlier removal of anomalous high-area/low-price observations reduces model sensitivity to leverage points.
- The final prediction combines stacked (Ridge meta-learner on OOF predictions) and blended (weighted average) ensembles for maximum stability.

## 9. References
- Competition: [House Prices - Advanced Regression Techniques](https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques)
- Dataset: Ames Housing Dataset, compiled by Dean De Cock for data science education
- Citation: Anna Montoya and DataCanary, House Prices - Advanced Regression Techniques, Kaggle, 2016
- Libraries: scikit-learn, XGBoost, LightGBM, Pandas, NumPy, Matplotlib, Seaborn